# 🔗 LangChain — Study & Revision Notebook

This notebook covers the core concepts of **LangChain**, a framework for building applications powered by LLMs. Each section demonstrates a key concept with practical examples.

**Topics covered:**
1. Direct API calls (OpenAI & Groq SDKs)
2. LangChain Model Wrappers (`ChatOpenAI`, `ChatGroq`)
3. Messages (`SystemMessage`, `HumanMessage`)
4. Output Parsers (`StrOutputParser`)
5. Prompt Templates (`ChatPromptTemplate`)
6. Chains using LCEL (LangChain Expression Language)
7. Sequential Chains
8. Coercion Functions (lambdas in chains)
9. `RunnableParallel` — run chains concurrently
10. `RunnablePassthrough` — pass data through while enriching it
11. LangSmith Tracing (observability)

---
## 1. Setup & API Keys
Install the required packages and load API keys. We use **OpenAI** and **Groq** as LLM providers throughout this notebook.

In [13]:
%pip install openai groq
%pip install --pre -U langchain langchain-openai langchain-Groq

In [14]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API')
GROQ_API_KEY = userdata.get('GROQ_API')

---
## 2. Direct API Calls (Without LangChain)
Before using LangChain, it's important to understand the **raw SDK** approach. Here we call OpenAI and Groq APIs directly using their Python clients. This shows the boilerplate you'd normally write — LangChain abstracts this away.

**Key takeaway:** Each provider has its own SDK and response format, making it hard to switch between models. LangChain solves this with a unified interface.

In [15]:
import openai
import os

openai_instance = openai.OpenAI(api_key=OPENAI_API_KEY)
openai_response = openai_instance.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=1,
    messages=[
        {"role": "system", "content": "You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."},
        {"role": "user", "content": "how many Egyptian football won African Cup?"}
        ]
    )
openai_response

ChatCompletion(id='chatcmpl-D65vkNc9U16MX6fSu1Mv9IWkVo2V7', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Egyptian football national team has won the African Cup of Nations seven times. One of the most memorable moments was in the 2006 final when Egypt faced Ivory Coast. The match went into a penalty shootout, and in a tense moment, goalkeeper Essam El-Hadary saved a crucial penalty to secure Egypt's victory.\n\nAnother unforgettable moment was in the 2008 final against Cameroon. Egypt scored late in the match with a stunning long-range strike from Mohamed Aboutrika, securing their record-breaking sixth African Cup of Nations title.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1770345136, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=108, prompt_tokens=43, total_tokens=151

In [16]:
openai_response.choices[0].message.content

"Egyptian football national team has won the African Cup of Nations seven times. One of the most memorable moments was in the 2006 final when Egypt faced Ivory Coast. The match went into a penalty shootout, and in a tense moment, goalkeeper Essam El-Hadary saved a crucial penalty to secure Egypt's victory.\n\nAnother unforgettable moment was in the 2008 final against Cameroon. Egypt scored late in the match with a stunning long-range strike from Mohamed Aboutrika, securing their record-breaking sixth African Cup of Nations title."

In [17]:
import groq

groq_instance = groq.Groq(api_key=GROQ_API_KEY)
groq_response = groq_instance.chat.completions.create(
    model = "llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."},
        {"role": "user", "content": "how many Egyptian football won African Cup?"}
        ]
)

groq_response

ChatCompletion(id='chatcmpl-14f00f41-429a-4e85-a37e-8a48bd8b4a4e', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="A great question about the rich history of Egyptian football! Egypt, also known as the Pharaohs, has an impressive record in the African Cup of Nations.\n\nTo date, Egypt has won the African Cup of Nations a total of 7 times:\n\n1. 1957 African Cup of Nations: Egypt won the inaugural tournament in Sudan, led by the legendary Mahmoud El-Gohary and Taha Ismail.\n2. 1959 African Cup of Nations: Egypt defended their title in Egypt, with Mahmoud El-Gohary scoring 5 goals in the tournament, including a hat-trick in the final against Sudan.\n3. 1962 African Cup of Nations: Egypt won the tournament in Ethiopia, with a 2-1 victory over Ethiopia in the final.\n4. 1986 African Cup of Nations: Egypt won the tournament in Egypt, with an unforgettable match featuring a 4-0 win over Morocco in the final, and also a dramatic moment wher

In [18]:
groq_response.choices[0].message.content

"A great question about the rich history of Egyptian football! Egypt, also known as the Pharaohs, has an impressive record in the African Cup of Nations.\n\nTo date, Egypt has won the African Cup of Nations a total of 7 times:\n\n1. 1957 African Cup of Nations: Egypt won the inaugural tournament in Sudan, led by the legendary Mahmoud El-Gohary and Taha Ismail.\n2. 1959 African Cup of Nations: Egypt defended their title in Egypt, with Mahmoud El-Gohary scoring 5 goals in the tournament, including a hat-trick in the final against Sudan.\n3. 1962 African Cup of Nations: Egypt won the tournament in Ethiopia, with a 2-1 victory over Ethiopia in the final.\n4. 1986 African Cup of Nations: Egypt won the tournament in Egypt, with an unforgettable match featuring a 4-0 win over Morocco in the final, and also a dramatic moment where Egypt's goalkeeper Ahmed Shobier saved a penalty shot in the semi-final against Cameroon.\n\nTwo memorable match moments that come to mind:\n\n1. 1998 African Cup of

---
## 3. LangChain Model Wrappers (`ChatOpenAI` & `ChatGroq`)
LangChain provides **unified wrappers** around different LLM providers. Instead of using each provider's SDK, you use `ChatOpenAI` or `ChatGroq` — both share the same `.invoke()` interface.

**Why it matters:** You can swap models by changing one line of code. The rest of your pipeline stays the same.

In [19]:
from langchain_openai import ChatOpenAI

gpt_model = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=OPENAI_API_KEY
    )
gpt_response = gpt_model.invoke("what is the captial of Egypt?")

gpt_response

AIMessage(content='Cairo', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 15, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-D65vmj3OHICGUvylVun4z2PSIWZfy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c30cb-07ba-7d40-b229-5f893c73d503-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 2, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [20]:
gpt_response.content

'Cairo'

In [21]:
from langchain_groq import ChatGroq

groq_model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY
)

groq_response = groq_model.invoke("what is the captial of Egypt?")

groq_response

AIMessage(content='The capital of Egypt is Cairo.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 43, 'total_tokens': 51, 'completion_time': 0.00625789, 'completion_tokens_details': None, 'prompt_time': 0.00305668, 'prompt_tokens_details': None, 'queue_time': 0.044128709, 'total_time': 0.00931457}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6b5c123dd9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c30cb-0c5e-7353-b303-721ddba7396e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 8, 'total_tokens': 51})

In [22]:
groq_response.content

'The capital of Egypt is Cairo.'

---
## 4. Messages (`SystemMessage` & `HumanMessage`)
LangChain uses typed **message objects** instead of raw dictionaries. `SystemMessage` sets the LLM's behavior/persona, and `HumanMessage` carries the user's input.

**Key benefit:** Cleaner, type-safe code. The same message list works across all chat model wrappers.

In [23]:
# Messages
from langchain_core.messages import HumanMessage, SystemMessage

input_Messages = [
    SystemMessage(content="You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
    HumanMessage(content="how many Egyptian football won African Cup?")
]

gpt_response = gpt_model.invoke(input_Messages)

gpt_response

AIMessage(content='Egyptian national team has won the African Cup of Nations 7 times in total. One of the most memorable moments was in 1986 when Egypt defeated Cameroon in penalties in the final. Another iconic moment was in 2006 when Egypt triumphed over Ivory Coast in the final with a last-minute goal from Mohamed Aboutrika, securing their 5th title.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 43, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-D65vnZqjDfN5AmDowZhN8QEQvUgX6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c30cb-0d72-7f10-a731-6d4ee6c76f7e-0', tool_calls=[], invalid_tool_calls

In [24]:
gpt_response.content

'Egyptian national team has won the African Cup of Nations 7 times in total. One of the most memorable moments was in 1986 when Egypt defeated Cameroon in penalties in the final. Another iconic moment was in 2006 when Egypt triumphed over Ivory Coast in the final with a last-minute goal from Mohamed Aboutrika, securing their 5th title.'

In [25]:
groq_model.invoke(input_Messages).content

"As an Egyptian TV football news analyst assistant, I'm delighted to share with you the proud history of the Egyptian national football team in the African Cup of Nations. The Egyptians have won the African Cup of Nations a total of 7 times:\n\n1. 1957: Egypt won the inaugural African Cup of Nations title, defeating Ethiopia 4-0 in the final.\n2. 1959: Egypt successfully defended their title, defeating Sudan 2-1 in the final.\n3. 1962: Egypt won their third African Cup of Nations title, defeating Ethiopia 4-0 in the final.\n4. 1986: Egypt, led by the legendary Mahmoud El Khatib, won their fourth title, defeating Morocco 1-0 in the final.\n5. 1998: Egypt, under the guidance of Mahmoud El Gohary, won their fifth title, defeating South Africa 2-0 in the final.\n6. 2006: Egypt, led by the skilled Emad Moteab, won their sixth title, defeating Ivory Coast 4-2 in a penalty shootout after the match ended 0-0.\n7. 2010: Egypt, with a star-studded team including Mohamed Aboutrika and Emad Moteab

---
## 5. Output Parsers (`StrOutputParser`)
The LLM returns an `AIMessage` object. An **Output Parser** extracts just the content you need. `StrOutputParser` converts the response to a plain string.

**Use case:** Essential when chaining — downstream components usually expect a string, not the full message object.

In [26]:
# parser
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

parser_gpt_response = output_parser.invoke(gpt_response)

parser_gpt_response

'Egyptian national team has won the African Cup of Nations 7 times in total. One of the most memorable moments was in 1986 when Egypt defeated Cameroon in penalties in the final. Another iconic moment was in 2006 when Egypt triumphed over Ivory Coast in the final with a last-minute goal from Mohamed Aboutrika, securing their 5th title.'

---
## 6. Prompt Templates (`ChatPromptTemplate`)
**Prompt Templates** let you define reusable prompts with **placeholder variables** (e.g., `{team_nationality}`). At runtime, you fill in the variables with `.invoke()`.

**Components:**
- `SystemMessagePromptTemplate` — template for the system message
- `HumanMessagePromptTemplate` — template for the user message
- `ChatPromptTemplate.from_messages()` — combines them into a full prompt

In [27]:
# prompt template

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
        HumanMessagePromptTemplate.from_template("how many {team_nationality} football team won {continental} Cup?")

    ]
)

prompt_template

ChatPromptTemplate(input_variables=['continental', 'team_nationality'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['continental', 'team_nationality'], input_types={}, partial_variables={}, template='how many {team_nationality} football team won {continental} Cup?'), additional_kwargs={})])

In [28]:
prompt_template_var = prompt_template.invoke(
    {"team_nationality": "Egyptian", "continental": "African"}
    )
prompt_template_var

ChatPromptValue(messages=[SystemMessage(content='You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments.', additional_kwargs={}, response_metadata={}), HumanMessage(content='how many Egyptian football team won African Cup?', additional_kwargs={}, response_metadata={})])

In [29]:
prompt_template_response = gpt_model.invoke(prompt_template_var)
prompt_template_response

AIMessage(content="Egyptian National Team has won the African Cup of Nations a record seven times. One of the most memorable moments was in 2006 when Egypt defeated Ivory Coast in the final with goalkeeper Essam El Hadary's heroic saves in the penalty shootout. Another memorable moment was in 2010 when Egypt lifted the trophy for the third consecutive time, beating Ghana 1-0 in the final with Mohamed Gedo scoring the winning goal in extra time.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 44, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-D65vq8O66AzvV0JRFllndiCABLFEn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': 

In [30]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

output_parser.invoke(prompt_template_response)

"Egyptian National Team has won the African Cup of Nations a record seven times. One of the most memorable moments was in 2006 when Egypt defeated Ivory Coast in the final with goalkeeper Essam El Hadary's heroic saves in the penalty shootout. Another memorable moment was in 2010 when Egypt lifted the trophy for the third consecutive time, beating Ghana 1-0 in the final with Mohamed Gedo scoring the winning goal in extra time."

---
## 7. Chains — LCEL (LangChain Expression Language)
**LCEL** uses the **pipe operator (`|`)** to chain components together: `prompt | model | parser`. This is the core pattern in LangChain.

```
prompt_template | llm_model | output_parser
```

Below we build **3 sequential chains**: information → translation → social media post. The output of each chain feeds into the next one.

In [31]:
# Chain
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

gpt_model = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=OPENAI_API_KEY
    )
info_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
        HumanMessagePromptTemplate.from_template("how many {team_nationality} football team won {continental} Cup?")
    ]
)

out_parser = StrOutputParser()

information_chain = info_prompt_template | gpt_model | out_parser

information_out = information_chain.invoke(
    {"team_nationality": "Egyptian", "continental": "African"}
)

print("========= info chain =========")
print(information_out)

egyption_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV translator assistant. Your job is to translate english from footballer analyzer to Egyptian Arabic Audience."),
        HumanMessagePromptTemplate.from_template("Analyzed topic: {information_chain_out}")
    ]
)

egyption_chain = egyption_prompt_template | gpt_model | out_parser

egyption_out = egyption_chain.invoke(
    {"information_chain_out": information_out}
)

print("========= Egyption chain =========")
print(egyption_out)

facebook_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian A journalist who writes articles for a sports newspaper facebook page."),
        HumanMessagePromptTemplate.from_template("geneate a sport post about {egyption_chain_out}")
    ]
)

facebook_chain = facebook_prompt_template | gpt_model | out_parser

facebook_out = facebook_chain.invoke(
    {"egyption_chain_out": egyption_out}
)

print("========= Facebook chain =========")
print(facebook_out)

========= info chain =========
Egyptian national football team has won the African Cup of Nations 7 times. One of the most memorable moments was in 2006 when Egypt won the tournament in front of their home fans, defeating the Ivory Coast in a dramatic penalty shootout in the final.

Another iconic moment was in 2010 when Egypt clinched their third consecutive title, a record in the tournament's history, by defeating Ghana 1-0 in the final with a late goal from Gedo. These victories have solidified Egypt's status as one of the most successful teams in African football history.
========= Egyption chain =========
تحليل الموضوع: منتخب مصر الوطني لكرة القدم فاز بكأس أمم أفريقيا 7 مرات. كانت لحظة من أبرز اللحظات في عام 2006 عندما فازت مصر بالبطولة أمام جماهيرها في المنزل، بعدما هزمت ساحل العاج في النهائي بركلات الترجيح في مشهد درامي.

لحظة أخرى أيقونية كانت في عام 2010 عندما حققت مصر لقبها الثالث على التوالي، وهو رقم قياسي في تاريخ البطولة، بعدما تغلبت على غانا 1-0 في النهائي بفضل هدف متأخر 

---
## 8. Language Detection & Conditional Routing
A practical example of using **multiple chains with conditional logic**. First, a chain detects the language of the user's question (returns ISO code like `en` or `ar`). Then, based on the result, we route to either an English chain or an Arabic chain.

**Pattern:** `detect language → if/else → invoke the right chain`. This is manual routing — later, LangGraph provides more elegant branching.

In [32]:
question = "how many Egyptian football team won African Cup?"
# question = "كم مرة فاز المنتخب المصري لكرة القدم ببطولة الامم الافريقية؟"

In [33]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

get_lang_model = ChatGroq(
    model = "moonshotai/kimi-k2-instruct",
    api_key=GROQ_API_KEY
)

get_lang_parser = StrOutputParser()

get_lang_prompt =ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            "You are a linguistic analyst. your job is to read user question and reply ONLY with langue ISO language Code, for example: ar for arabic, en for english."
        ),
        HumanMessagePromptTemplate.from_template(
            "user question: {question}"
            )
    ]
)

get_lang_chain = get_lang_prompt | get_lang_model | get_lang_parser

language = get_lang_chain.invoke(question)

print("========= language  =========")
print(language)

en_model =ChatGroq(
    model="moonshotai/kimi-k2-instruct",
    api_key=GROQ_API_KEY
)

en_parser = StrOutputParser()

en_prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            "You are ENGLISH language AI Assistant answer question in English language ONLY."
            ),
        HumanMessagePromptTemplate.from_template(
            "user question: {question}"
            )
        ]
)

en_chain = en_prompt | en_model | en_parser

ar_model =ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY
)

ar_parser = StrOutputParser()

ar_prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template
         ("You are ARABIC language AI Assistant answer question in Arabic language ONLY."
         ),
        HumanMessagePromptTemplate.from_template(
            "user question: {question}"
            )
        ]
)

ar_chain = ar_prompt | ar_model | ar_parser


if language == "en":
    print("========= English Response =========")
    print(en_chain.invoke(question))
elif language == "ar":
    print("========= Arabic Response =========")
    print(ar_chain.invoke(question))

========= language  =========
en
========= English Response =========
Egypt has won the Africa Cup of Nations (AFCON) **7 times** — more than any other country.


---
## 9. LangSmith Tracing (Observability)
**LangSmith** is LangChain's observability platform. By setting environment variables, every chain invocation is automatically traced — you can inspect inputs, outputs, latency, and token usage in the LangSmith dashboard.

**Setup:** Set `LANGCHAIN_TRACING_V2=true`, provide your API key, and specify a project name.

In [34]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API')
GROQ_API_KEY = userdata.get('GROQ_API')

In [ ]:
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# Replace 'your_langchain_api_key_here' with your actual LangChain/LangSmith API key
os.environ["LANGCHAIN_API_KEY"] = "your_langchain_api_key_here"  # userdata.get('LANGCHAIN_API') 
os.environ["LANGCHAIN_PROJECT"] = "test"

In [36]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

groq_model = ChatGroq(
    model="moonshotai/kimi-k2-instruct",
    api_key=GROQ_API_KEY
    )
prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
        HumanMessagePromptTemplate.from_template("how many {team_nationality} football team won {continental} Cup?")
        ]
    )
Parser = StrOutputParser()
chain = prompt_template | groq_model | Parser
respond = chain.invoke({"team_nationality": "Egyptian", "continental": "African"})
respond

'Egypt lifts the trophy more times than any other nation on the continent:  \n7 titles – 1957, 1959, 1986, 1998, 2006, 2008, 2010.\n\nTwo moments every Egyptian still replays in his head:\n\n1. Cairo Stadium, 22 February 1986 – Ghana are the “bogey side” of the ’80s and the final is locked at 0-0. In the 103rd minute, a spindly 19-year-old centre-back called Magdi Abdel-Ghani ghosts in at the far post to head the golden-goal that ends Egypt’s 27-year drought and sends 100,000 fans spilling onto the grass in tears.\n\n2. Accra Sports Stadium, 10 February 2010 – Egypt are supposedly “too old” and Ghana are the host favourites. With the score 0-0 and the clock ticking, Mohamed “Bobo” Nagy collects a loose ball 30 metres out, shapes as if to cross, then lashes an unstoppable left-foot rocket into the top corner. 1-0, game flipped, and Egypt march to a record third straight AFCON crown.'

---
## 10. Coercion Functions (Lambdas in Chains)
You can inject **Python functions (lambdas)** into a chain to transform input before it reaches the prompt. Here, a plain string like `"Egypt"` is converted into `{"team_nationality": "Egyptian", "continental": "African"}` using helper functions.

**Pattern:** Use a `dict` with lambda values as the first step of a chain to map raw input into template variables.

In [37]:
# coercion functions

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

groq_model = ChatGroq(
    model="moonshotai/kimi-k2-instruct",
    api_key=GROQ_API_KEY
    )
prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
        HumanMessagePromptTemplate.from_template("how many {team_nationality} football team won {continental} Cup?")
        ]
    )
Parser = StrOutputParser()

def nationality (country: str):
    if country == "Egypt":
        return "Egyptian"
    elif country == "Italy":
        return "Italian"
    else:
        return "Unknown"

def cup_nation (country: str):
    if country == "Egypt":
        return "African"
    elif country == "Italy":
        return "Euro"
    else:
        return "Unknown"

coercion_chain = (
    {
        "team_nationality": lambda x: nationality(x),
        "continental": lambda x: cup_nation(x)
    }
    | prompt_template
    | groq_model
    | Parser
)

coercion_response = coercion_chain.invoke("Egypt")
coercion_response

'The Pharaohs tower over Africa with **7** titles:  \n- **1957**  \n- **1959**  \n- **1986**  \n- **1998**  \n- **2006**  \n- **2008**  \n- **2010**  \n\nTwo moments every Egyptian still replays in his sleep:  \n1. **2006 final in Cairo**: Hassan’s 57-minute header vs Ivory Coast, and that 23-penalty shoot-out where Abdel-Wahed lifts it 12 yards down the corridor of fame.  \n2. **2010 final in Luanda**: Gedo’s 85th-minute dagger into the side of Ghana that turned the clock to 7-0 for the continent.'

---
## 11. `RunnableParallel` — Run Chains Concurrently
`RunnableParallel` executes **multiple chains at the same time** and collects their outputs into a dict. Here, two different LLMs write poems in parallel, then a third LLM evaluates which poem is better.

**Flow:**
```
Input → RunnableParallel(os_poem=chain_A, cs_poem=chain_B) → eval_prompt → judge_model → result
```

**Key benefit:** Saves time by running independent tasks concurrently instead of sequentially.

In [38]:
# Runnable Parallels Chains
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel


poem_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            """
You are a skilled Arabic poet in a poetry competition.
When given a country, write a titled poem in classical Arabic that follows poetic meter and rhyme.
The poem should highlight the country’s heritage, nature, and modern culture, inspiring admiration and tourism.
Respond only in Arabic and DO NOT add translate your response to English.
            """
            ),
        HumanMessagePromptTemplate.from_template(
            "Write a poem about country: {country}"
            )
        ]
    )

parser = StrOutputParser()

os_openai = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY
)
cs_openai = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=OPENAI_API_KEY
)

os_chain = poem_prompt_template | os_openai | parser
cs_chain = poem_prompt_template | cs_openai | parser


eval_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            "You are a jury member in an Arabic poetry competition. Compare the poems decide and announce the winner name who wrote the best poem for promoting tourism. Respond only in Arabic."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            opensource Poem :
            {os_poem}

            closesource Poem :
            {cs_poem}
            """
            )
        ]
)

eval_model = ChatGroq(
    model="moonshotai/kimi-k2-instruct",
    api_key=GROQ_API_KEY
    )

eval_chain = (
    RunnableParallel(
        os_poem=os_chain,
        cs_poem=cs_chain
    )
    | eval_prompt_template
    | eval_model
    | parser
)

evaluation_out = eval_chain.invoke({"country": "Egypt"})
evaluation_out

'يا أهلَ السِّرى والنورِ، يا نُزَاعَيْنِ في حضارةِ النيلِ:\n\nلقد اقتربنا من النهرِ المتكرِّفِ، واستوقفنا أهرامَهُ، فتبادلتْ أجنحةُ الزمانِ فيها النبيحَ.\n\nنقرأ في "مصرُ الأمان" انسجامًا حيًّا من صورٍ تَتْقِي: الأهراماتُ تُنادي، والنيلُ يَجري، فتُدوِّرُ الأمانَ في أذنِ السائحِ. لكنها تُقفَلُ على دائرةٍ واحدةٍ من الحنينِ، فلا تتجاوزُ السلامةَ والجمالَ إلى حركةِ السفرِ المباشرة.\n\nأمّا "في بلاد الفراعنة مجدٌ عالٍ" فهي تصلُ إلى يدِ المسافِرِ قبلَ أن يُكملَ تحيَّةَ الصباحِ، فتُحيِّي وتُرحِّب وتُصرِفُ فيه دعوةً مفتوحةً: "فاسحبوا إلى مصر يا أححباء، تمتعوا، استمتعوا، أهلاً وسهلًا". إنها تُعلنُ التسويقَ بوضوحٍ، تغنّي لكلّ مدينةٍ، تُشير إلى كلّ معلمةٍ، تُفصلُ العنوانِ، فتُصبحُ دعوةً مباشرةً شافيةً.\n\nفنحن نقرّر أنّ الفوزَ بالنزّيةِ يذهب إلى:  \n**"في بلادِ الفراعنةِ مجدٌ عالٍ"**  \nلأنّها تُنبِتُ في قلبِ القارئِ نداءَ السفرِ، لا مجردَ التأمُّلِ.'

---
## 12. `RunnablePassthrough` — Enrich Data While Passing It Through
`RunnablePassthrough.assign()` lets you **add new keys** to the data dict while keeping all existing keys intact. Unlike `RunnableParallel` alone (which replaces the dict), `assign()` preserves the original outputs and appends new ones.

**Use case here:** After generating two poems in parallel, we keep both poems AND add the judge's evaluation — then a final lambda formats everything into one readable string.

In [39]:
# RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


poem_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            """
You are a skilled Arabic poet in a poetry competition.
When given a country, write a titled poem in classical Arabic that follows poetic meter and rhyme.
The poem should highlight the country’s heritage, nature, and modern culture, inspiring admiration and tourism.
Respond only in Arabic and DO NOT add translate your response to English.
            """
            ),
        HumanMessagePromptTemplate.from_template(
            "Write a poem about country: {country}"
            )
        ]
    )

parser = StrOutputParser()

os_openai = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=GROQ_API_KEY
)
cs_openai = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=OPENAI_API_KEY
)

os_chain = poem_prompt_template | os_openai | parser
cs_chain = poem_prompt_template | cs_openai | parser


eval_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
            "You are a jury member in an Arabic poetry competition. Compare the poems decide and announce the winner name who wrote the best poem for promoting tourism. Respond only in Arabic."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            opensource Poem :
            {os_poem}

            closesource Poem :
            {cs_poem}
            """
            )
        ]
)

eval_model = ChatGroq(
    model="moonshotai/kimi-k2-instruct",
    api_key=GROQ_API_KEY
    )

eval_chain = (
    RunnableParallel(
        os_poem=os_chain,
        cs_poem=cs_chain
    )

    |RunnablePassthrough.assign(
    poems = eval_prompt_template| eval_model| parser
    )
    |(
        lambda inputs: f"open source Poem:\n{inputs['os_poem']}\n\n"
                      f"close source Poem:\n{inputs['cs_poem']}\n\n"
                      f"Evalution:\n{inputs['poems']}"
    )

)

evaluation_out = eval_chain.invoke({"country": "Egypt"})
evaluation_out

'open source Poem:\n**مصرُ الروحِ والضياءِ**\n\nيا مِصرُ يا مِهدَ الفُنونِ والعُلا  \nفيكِ النيلُ يَجري كَسَيلِ الحُلا  \nأهرامُكِ تَستقبلُ شمسَ الصباحِ سَنا  \nوالمِقْصَرُ يَحكي مَجدًا لا يَفنى  \n\nالقلعةُ في أسوانِ تحكي أساطيرَ الزمانا  \nوالقاهرةُ صَفْحَتُها سِحرٌ في كلِّ مكانا  \nفي شوارعِها تُغنّى الألحانُ والأنغاما  \nوالشبابُ يَبني مستقبلًا بِنورِ الإبداعا  \n\nالبحرُ الأحمرُ يَستضيفُ سُفُنَ السَّفَرِ فضا  \nوأرضُ الصحراءِ تزهو بِنُجومٍ سَحرا  \nيا زائرًا هَنا سَتَجدُ عِطرَ الحكايا  \nفَتَسْكُنُ فِي قَلبِكِ مِصرُ أبدًا خَلا\n\nclose source Poem:\nفي مصر العريقة تاريخٌ عريقْ، \nبالأهرامات تحكي قصةً وتواريقْ، \n\nأرضُ الفراعنة بأنقوشٍ وتماثيلْ، \nتبهرُ العقولَ بجمالِها الفاتنِ الجميلْ، \n\nنيلها الخصبُ يروي الأرضَ ويبتهجْ، \nوأهلُها الكرماءُ بأخلاقٍ يتزينون بها ويتجلى شجٌّ وخيلْ، \n\nفي كلِّ شارعٍ ترى تاريخَها متجليًا، \nوبأسواقها الضاربةِ بالأصالة تلتقي الحداثةُ بالتقاليدِ في حضنٍ واحدٍ جميلْ، \n\nأهلاً بكم في مصرَ العظيمةِ، \nحيثُ التاريخُ يشدو، والطبيعةُ تغني، والثقافةُ تتوارث